# Gym Pose Project

This section builds a complete end-to-end project for one exercise (default: squat):
1. Capture training images from webcam.
2. Extract MediaPipe pose features.
3. Train a classifier for correct vs incorrect form.
4. Run real-time prediction with live feedback.

In [ ]:
# Install extras if needed (uncomment if your kernel is missing packages)
# !pip install mediapipe pandas joblib scikit-learn

import os
import glob
from pathlib import Path
from collections import Counter, deque

import cv2
import joblib
import mediapipe as mp
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
EXERCISE = "squat"
LABELS = ["correct", "incorrect"]
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / EXERCISE
FEATURES_CSV = PROJECT_ROOT / "data" / "features" / f"{EXERCISE}_features.csv"
MODEL_PATH = PROJECT_ROOT / "models" / f"{EXERCISE}_form_rf.joblib"

for label in LABELS:
    (DATA_ROOT / label).mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "models").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data" / "features").mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folders ready:")
for label in LABELS:
    print("-", DATA_ROOT / label)

2026-04-07 20:34:28.130076: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-07 20:34:28.137309: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-07 20:34:28.146731: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-07 20:34:28.150070: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-07 20:34:28.157287: I tensorflow/core/platform/cpu_feature_guar

Data folders ready:
- dataset/squat/correct
- dataset/squat/incorrect


In [ ]:
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils


def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360.0 - angle
    return angle


def pose_to_feature_vector(landmarks):
    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32)

    hip_center = (coords[23] + coords[24]) / 2.0
    shoulder_dist = np.linalg.norm(coords[11] - coords[12]) + 1e-6
    norm_coords = (coords - hip_center) / shoulder_dist

    left_knee = calculate_angle(coords[23][:2], coords[25][:2], coords[27][:2])
    right_knee = calculate_angle(coords[24][:2], coords[26][:2], coords[28][:2])
    left_hip = calculate_angle(coords[11][:2], coords[23][:2], coords[25][:2])
    right_hip = calculate_angle(coords[12][:2], coords[24][:2], coords[26][:2])

    torso_vec = coords[11] - coords[23]
    torso_incline = np.degrees(np.arctan2(torso_vec[1], torso_vec[0]))

    engineered = np.array([
        (left_knee + right_knee) / 2.0,
        (left_hip + right_hip) / 2.0,
        torso_incline,
        np.linalg.norm(coords[23] - coords[24]) / shoulder_dist,
    ], dtype=np.float32)

    return np.concatenate([norm_coords.flatten(), engineered], axis=0)

Feature utilities loaded.


In [ ]:
def collect_samples(label, target_count=200, camera_index=0):
    save_dir = os.path.join(DATA_ROOT, label)
    existing = len(glob.glob(os.path.join(save_dir, "*.jpg")))
    count = existing

    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam")

    print(f"Collecting for label='{label}'.")
    print("Keys: [c] capture, [q] quit")

    while cap.isOpened() and count < (existing + target_count):
        ret, frame = cap.read()
        if not ret:
            break

        view = frame.copy()
        cv2.putText(view, f"Label: {label}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
        cv2.putText(view, f"Saved: {count - existing}/{target_count}", (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.putText(view, "Press 'c' to capture, 'q' to stop", (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.imshow("Dataset Capture", view)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("c"):
            out_path = os.path.join(save_dir, f"{label}_{count:05d}.jpg")
            cv2.imwrite(out_path, frame)
            count += 1
        elif key == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"Done. Total files now in {save_dir}: {len(glob.glob(os.path.join(save_dir, '*.jpg')))}")

In [ ]:
def build_feature_csv(output_csv=FEATURES_CSV):
    rows = []
    skipped = 0

    with mp_pose.Pose(static_image_mode=True, model_complexity=1) as pose:
        for label in LABELS:
            image_paths = glob.glob(os.path.join(DATA_ROOT, label, "*.jpg"))
            for image_path in image_paths:
                image = cv2.imread(image_path)
                if image is None:
                    skipped += 1
                    continue

                rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                result = pose.process(rgb)

                if not result.pose_landmarks:
                    skipped += 1
                    continue

                vec = pose_to_feature_vector(result.pose_landmarks.landmark)
                row = {f"f{i}": float(v) for i, v in enumerate(vec)}
                row["label"] = label
                rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No valid pose samples found. Collect more clear images first.")

    df.to_csv(output_csv, index=False)
    print(f"Saved features: {output_csv}")
    print("Rows:", len(df), "Skipped:", skipped)
    print(df["label"].value_counts())

In [ ]:
def train_form_classifier(features_csv=FEATURES_CSV, model_path=MODEL_PATH):
    df = pd.read_csv(features_csv)
    X = df.drop(columns=["label"])
    y = df["label"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    clf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    )
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred, labels=LABELS))

    bundle = {
        "model": clf,
        "labels": LABELS,
        "exercise": EXERCISE,
        "feature_count": X.shape[1],
    }
    joblib.dump(bundle, model_path)
    print(f"Saved model: {model_path}")

In [ ]:
def run_realtime_form_check(model_path=MODEL_PATH, camera_index=0, smooth_window=15):
    bundle = joblib.load(model_path)
    clf = bundle["model"]
    labels = bundle["labels"]

    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam")

    pred_history = deque(maxlen=smooth_window)

    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as pose:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = pose.process(rgb)

            status_text = "No pose detected"
            confidence = 0.0
            color = (0, 0, 255)

            if result.pose_landmarks:
                mp_draw.draw_landmarks(frame, result.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                vec = pose_to_feature_vector(result.pose_landmarks.landmark)
                probs = clf.predict_proba([vec])[0]
                idx = int(np.argmax(probs))
                pred = labels[idx]
                confidence = float(probs[idx])
                pred_history.append(pred)

                smooth_pred = Counter(pred_history).most_common(1)[0][0]
                status_text = f"{smooth_pred.upper()} ({confidence:.2f})"
                color = (0, 220, 0) if smooth_pred == "correct" else (0, 140, 255)

            cv2.rectangle(frame, (0, 0), (480, 85), (0, 0, 0), -1)
            cv2.putText(frame, f"Exercise: {EXERCISE}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            cv2.putText(frame, status_text, (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.85, color, 2)
            cv2.imshow("Gym Form Check", frame)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()

In [12]:
#collect_samples("correct", target_count=250)
#collect_samples("incorrect", target_count=250)
#build_feature_csv()
#train_form_classifier()
run_realtime_form_check()

I0000 00:00:1775558350.667811  105867 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1775558350.669730  109140 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: Mesa Intel(R) Graphics (ADL GT2)
W0000 00:00:1775558350.717900  109123 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775558350.733730  109134 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/idozii/miniconda3/envs/ds_env/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/home/idozii/miniconda3/en

KeyboardInterrupt: 